# What the augmentation does to a window

Every panel draws the window the way the labelling app does: the phase labels are the
**shading over the signal**, and the same labels repeat as a **ribbon underneath** where
boundaries are easier to read off. Colours are `plot.phase_colors` from the config, so a
window here looks like the same window in the app.

The augmentation being shown is `training.augmentation` as configured - not a demo set.
Figures are written to `out/augmentation/`.

In [ ]:
# This repo is not installed as a package (`package-mode = false`), so it imports only with
# its root on the path. A kernel starts in the notebook's own directory, so put it there -
# found by walking up for pyproject.toml, so it works whatever the working directory is.
import sys
from pathlib import Path


def repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'phase').is_dir():
            return candidate
    raise RuntimeError(f'no inhale-exhale-training checkout at or above {start}')


REPO = repo_root(Path.cwd().resolve())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

try:
    import torch  # noqa: F401
except ModuleNotFoundError as missing:
    raise RuntimeError(
        f'{missing.name} is missing: this kernel is not the project venv. Register it with '
        f'`poetry run python -m ipykernel install --user --name inhale-exhale-training` '
        f'and select the inhale-exhale-training kernel.') from missing

print(f'repo root {REPO}')


In [ ]:

import numpy as np
import pandas as pd
from IPython.display import Image
from omegaconf import OmegaConf

import phase
from phase.dataset import (LengthBucketSampler, WindowDataset, baseline_drift, collate,
                           time_warp)
from phase.building import load_windows, resolve
from phase.figures import colors_of, plot_windows
from phase.labels import PHASES, class_counts

OUT = REPO / 'out' / 'augmentation'

root = OmegaConf.load(REPO / 'parameter' / 'config.yaml')
training = OmegaConf.load(REPO / 'parameter' / 'training' / 'default.yaml')
data_cfg = OmegaConf.load(REPO / 'parameter' / 'data' / 'phases.yaml')
AUG = OmegaConf.to_container(training.augmentation, resolve=True)
PLOT = OmegaConf.to_container(root.plot, resolve=True)

AUG

In [ ]:
# `data.dir` decides, not a name sort: `phases_human_...` sorts after every
# `phases_algorithm_...` whatever the stamps say. `resolve` reads the build stamp.
dataset = resolve(REPO / data_cfg.root, data_cfg.dir, data_cfg.labels.source)
manifest = load_windows(dataset)
train = manifest[manifest['split'] == 'train'].reset_index(drop=True)
FPS = float(manifest['analysis_fps'].iloc[0])

print(dataset.name)
print(f'{len(manifest)} windows, {len(train)} in train, {FPS:g} fps')
print(manifest['env'].value_counts().to_dict())

## The window everything below starts from

Picked as the train window carrying the most labelled spans, so there is plenty of phase
structure to watch move.

In [ ]:
plain = WindowDataset(train, dataset, crop=None, train=False, seed=root.seed)
pick = int(train['n_spans'].idxmax())
values, target = plain._window(pick)
row = train.iloc[pick]


def item(values, target, title):
    """Labels twice over: shaded on the signal, and as the ribbon below it."""
    return {'values': values, 'reference': target, 'prediction': target,
            'fps': FPS, 'title': title}


def draw(items, name, heading):
    """Writes the page to out/ and renders it inline - out/ is gitignored, so a
    markdown image link would be dead for anyone but the person who ran this."""
    out = plot_windows(items, OUT / name, heading=heading, plot_cfg=PLOT,
                       reference_name='labels', prediction_name='labels',
                       caption='shading = labels over the signal · ribbon = the same labels')
    return Image(filename=str(out))


def described(values, target):
    counts = class_counts(target)
    share = ' '.join(f'{n}={counts[n]}' for n in PHASES)
    return f'{values.size} samples, std {values.std():.2g} | {share}'


title = (f"window {int(row['RespirationWindowID'])} "
         f"({row['env']}, {int(row['n_spans'])} spans) - {described(values, target)}")
draw([item(values, target, title)], 'baseline.png', 'As stored')

`unknown` sits at the crest of most breaths. That is structural, not a gap in the
labelling - production emits no phase for the turn from inhale to exhale.

## Each augmentation against the window it came from

One row per augmentation, forced on at the configured strength. The original is the grey
line, the augmented one is black, and **both are scaled by the original's own statistics** -
so a change in amplitude or offset is visible instead of being normalised away. The shading
is the labels, which only `time_warp` moves.


In [ ]:
import matplotlib.pyplot as plt

from phase.figures import GRID, INK, INK_SOFT, SURFACE, legend_handles
from phase.labels import targets_to_spans

COLORS = colors_of(PLOT)


def compare(rows, name, heading):
    """Original against augmented on one shared scale, labels shaded behind."""
    base_mean, base_std = values.mean(), values.std() or 1.0
    norm = lambda v: (v - base_mean) / base_std
    span = max(np.abs(norm(np.concatenate([r['values'] for r in rows]))).max(), 3.0) * 1.1

    fig, axes = plt.subplots(len(rows), 1, figsize=(11, 1.9 * len(rows) + 1.1),
                             squeeze=False, facecolor=SURFACE)
    for ax, row in zip(axes.ravel(), rows):
        ax.set_facecolor(SURFACE)
        for s in targets_to_spans(np.asarray(row['target'])):
            ax.axvspan(s['start'] / FPS, s['end'] / FPS, facecolor=COLORS[s['phase']],
                       alpha=PLOT['span_alpha'], linewidth=0, zorder=0)
        ax.plot(np.arange(values.size) / FPS, norm(values), color='#b6b4b0',
                linewidth=2.4, zorder=2, label='as stored')
        ax.plot(np.arange(row['values'].size) / FPS, norm(row['values']),
                color=PLOT['trace_color'], linewidth=1.4, zorder=3, label='augmented')
        if row.get('show_added'):
            # What the augmentation actually put in, parked below the trace. Drift at
            # 0.12x and noise at 0.05x are otherwise too small to separate by eye.
            added = norm(row['values']) - norm(values)
            ax.plot(np.arange(added.size) / FPS, added - span * 0.62,
                    color='#c0392b', linewidth=1.0, zorder=4, label='what it added')
            ax.axhline(-span * 0.62, color=GRID, linewidth=0.6, zorder=1)
        ax.set_xlim(0, max(values.size, row['values'].size) / FPS)
        ax.set_ylim(-span, span)
        ax.set_yticks([])
        ax.grid(axis='x', color=GRID, linewidth=0.6)
        for side in ('top', 'right', 'left'):
            ax.spines[side].set_visible(False)
        ax.spines['bottom'].set_color(GRID)
        ax.tick_params(axis='x', colors=INK_SOFT, labelsize=7.5, length=0)
        ax.set_title(row['title'], fontsize=8.5, color=INK, loc='left', pad=6)
    axes.ravel()[-1].set_xlabel('seconds', fontsize=8, color=INK_SOFT)
    for ax in axes.ravel():
        ax.legend(loc='upper right', frameon=False, fontsize=7,
                  labelcolor=INK_SOFT, ncol=3)
    fig.suptitle(heading, fontsize=10.5, color=INK, x=0.012, ha='left', y=0.998)
    fig.legend(handles=legend_handles(COLORS), loc='lower center', ncol=4, frameon=False,
               fontsize=8, labelcolor=INK_SOFT, bbox_to_anchor=(0.5, -0.005))
    fig.tight_layout(rect=(0.01, 0.03, 1, 0.975), h_pad=2.4)
    out = OUT / name
    fig.savefig(out, dpi=160, facecolor=SURFACE, bbox_inches='tight')
    plt.close(fig)
    return Image(filename=str(out))


rng = np.random.default_rng(root.seed)
low, high = AUG['time_warp_range']
warped, warped_target = time_warp(values, target, (high, high), rng)

def delta(v):
    return f'{v.size} samples, std x{v.std() / values.std():.2f}'

compare([
    {'values': warped, 'target': warped_target,
     'title': f'time_warp x{high} (p={AUG["time_warp_prob"]}) - {delta(warped)}, '
              f'labels resampled with it'},
    {'values': values * high2 if (high2 := AUG['amplitude_range'][1]) else values,
     'target': target,
     'title': f'amplitude x{high2} (p={AUG["amplitude_prob"]}) - '
              f'{delta(values * high2)}, and z-scoring removes it again'},
    {'values': values + baseline_drift(values.size, AUG['drift_scale'], rng) * values.std(),
     'target': target,
     'show_added': True,
     'title': f'baseline drift {AUG["drift_scale"]} (p={AUG["drift_prob"]}) - '
              f'a slow wander under the breathing, no energy at breathing rate'},
    {'values': values + rng.normal(0, AUG['noise_scale'] * values.std(),
                                   values.size).astype(np.float32),
     'target': target,
     'show_added': True,
     'title': f'noise {AUG["noise_scale"]} of std (p={AUG["noise_prob"]}) - '
              f'jitter, shape untouched'},
], 'comparison.png', 'augmented against as-stored, on the original\'s scale')


Read down the rows:

- **time_warp** - the black line runs past the grey one; the shading stretches with it. This
  is the only row where the labels move, and the only one that changes the tensor's length.
- **amplitude** - visibly twice the swing here, and provably invisible to the model: the next
  section shows per-window z-scoring returns it to the original exactly.
- **drift** - the black line pulls away from the grey slowly, without adding anything at
  breathing rate. Boundaries stay where they were, so it teaches slope over absolute level.
- **noise** - rides on the grey line rather than departing from it.


## What a training epoch actually draws

Now through `WindowDataset` itself with `train=True`, so each draw composes the augmentations
at their configured probabilities. This is what the network sees.

In [ ]:
augmented = WindowDataset(train, dataset, crop=None, train=True, augment=AUG,
                          normalise=data_cfg.normalise, seed=root.seed)

draws = []
for n in range(6):
    got = augmented[pick]
    x = got['x'].numpy()[0]
    y = got['y'].numpy()
    draws.append(item(x, y, f'draw {n + 1} - {described(x, y)}'))

draw(draws, 'composite.png', 'six draws of one window, as configured')

Rows with a different sample count are the warp. The class counts move with it, because
resampling changes how many samples a phase spans - not which phase it is.


## Amplitude scaling does nothing at all

`WindowDataset` z-scores every window on its own statistics, because that is what the
device can do at inference - it has no corpus statistics. Scaling a window by a positive
constant and then z-scoring it returns the original exactly, so `amplitude_prob` and
`amplitude_range` cannot reach the model.

Worth measuring rather than reasoning about, because the config reads as though amplitude
robustness were covered.

In [ ]:
checks = []
for i in range(len(train)):
    v, _ = plain._window(i)
    base = plain._normalise(v.copy())
    for factor in (AUG['amplitude_range'][0], 1.0, AUG['amplitude_range'][1]):
        checks.append(np.abs(base - plain._normalise((v * factor).astype(np.float32))).max())

print(f'amplitude, max deviation after z-scoring: {max(checks):.3e}  <- identity')

rng2 = np.random.default_rng(root.seed)
v, _ = plain._window(0)
for name, altered in [
    ('drift', v + baseline_drift(v.size, AUG['drift_scale'], rng2) * v.std()),
    ('noise', v + rng2.normal(0, AUG['noise_scale'] * v.std(), v.size)),
]:
    moved = np.abs(plain._normalise(v.copy())
                   - plain._normalise(altered.astype(np.float32))).max()
    print(f'{name:9s} max deviation after z-scoring: {moved:.3e}  <- survives')

## Drift is scaled to the window, and was not

`baseline_drift` returns an amplitude around `drift_scale`. A raw radar window has a std
near `3e-4`, so passing `0.3` straight in put the wander **315x over** the breathing - the
trace became a slow sinusoid and the per-window z-scoring then compressed the actual
breathing into the bottom fraction of a percent of the range, while the phase labels stayed
exactly where they were.

`noise_scale` had always been applied as a fraction of `values.std()`; `drift_scale` had not.
It is now, which is what the docstring's "slow wander *under* the breathing" describes.

In [ ]:
raw = np.array([plain._window(i)[0].std() for i in range(len(train))])

probe = WindowDataset(train, dataset, augment={'drift_prob': 1.0,
                                              'drift_scale': AUG['drift_scale']})
ratio = []
for i in range(len(train)):
    v, t = plain._window(i)
    out, _ = probe._augment(v.copy(), t, np.random.default_rng(i))
    ratio.append((out - v).std() / (v.std() or 1.0))
ratio = np.array(ratio)

print(f'raw window std      median {np.median(raw):.2e}')
print(f'drift / signal std  median {np.median(ratio):.2f}  max {ratio.max():.2f}')
print(f'windows swamped     {int((ratio > 1).sum())} of {len(ratio)}')
print()
print(f'before the fix this ratio was 315 and every window was swamped,')
print(f'on {AUG["drift_prob"]:.0%} of the split every epoch.')

## The warp defeats the length bucketing

`LengthBucketSampler` groups windows by their **stored** length so a batch needs no padding,
because padding is masked out of the loss but is *not* hidden from `BatchNorm`, which
normalises over batch and length together and carries those statistics into inference.

`time_warp` resamples *after* the sampler has chosen the batch, so the realised lengths no
longer match the buckets and `collate` pads to the batch's longest member again.

In [ ]:
stored = train['samples'].to_numpy()
realised = np.array([augmented[i]['x'].shape[-1] for i in range(len(train))])


def padding_share(ds):
    sampler = LengthBucketSampler(stored, int(data_cfg.batch_size), shuffle=False,
                                  seed=root.seed)
    padded = total = 0
    for batch_rows in sampler._batches():
        batch = collate([ds[i] for i in batch_rows])
        padded += int((~batch['mask']).sum())
        total += int(batch['mask'].numel())
    return 100 * padded / total


print(f'stored lengths    {sorted(set(stored.tolist()))}')
print(f'realised lengths  {len(set(realised.tolist()))} distinct, '
      f'{realised.min()}-{realised.max()}')
print(f'length changed    {int((realised != stored).sum())} of {len(train)} windows\n')
print(f'padding, train  (augmented)   {padding_share(augmented):5.1f}%'
      f' of the stacked tensor')
print(f'padding, eval   (no augment)  {padding_share(plain):5.1f}%')


### What that means

The bucketing delivers its zero-padding guarantee on val and test, where nothing is
augmented. On train it does not: the warp reintroduces padding that `BatchNorm` sees.

Three ways out, none applied here - this notebook measures, it does not change training:

1. Warp to a **fixed output length** so the bucket still holds after augmentation.
2. Have the sampler bucket on the **realised** length by drawing the warp factor per batch
   rather than per window.
3. Drop `time_warp` and rely on the natural rate spread already in the pool.

Which is right depends on whether the warp earns its keep, and that is an ablation, not a
judgement call to make here.